In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR
from sklearn.metrics import root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
RANDOM_STATE = 42

## Для всех дальнейших экспериментов выделю стратифицированную подвыборку на 300к объектов
Иначе мне не хватит времени обучать каждую модель на 1.5М строк

In [ ]:
df_train = pd.read_csv("../train.csv")

Тк стратифицированная подвыборка работает только для классификации, поделю целевой признак на 100 классов

In [ ]:
_y = (df_train['deal_probability'].round(2) * 100).astype(int)

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=300000, random_state=42)

In [ ]:
rest_idx, small_idx = next(splitter.split(df_train, _y))
df_small = df_train.iloc[small_idx].reset_index(drop=True)

Отделяю по 10% валидационную и тестовую выборки, **отложенные по времени**, поскольку данные имеют временной признак (дата публикации)

In [ ]:
df_small['activation_date'] = pd.to_datetime(df_small['activation_date'])
df_small = df_small.sort_values(by='activation_date').reset_index(drop=True)

length = len(df_small.index)
train_end_id = int(0.8 * length)
valid_end_id = int(0.9 * length)

df_small_train = df_small.iloc[:train_end_id, :]
df_small_valid = df_small.iloc[train_end_id:valid_end_id, :]
df_small_test = df_small.iloc[valid_end_id:, :]

In [ ]:
print(df_small_train.shape)
print(df_small_valid.shape) 
print(df_small_test.shape)

(240000, 18)
(30000, 18)
(30000, 18)


## Проверяем базовые модели без доп настроек ГП
- линейная регрессия
- метод опорных векторов
- решающее дерево
- случайный лес

Сделаем общий препроцессинг:
- заполняю пропуски, 
- извлекаю день недели (тк там всего 21 день, конкретная дата не будет иметь значения), 
- OHE для категориальных признаков

In [ ]:
def fit_linear_pipeline(df_train):

    df = df_train.copy()

    num_cols = ['price', 'item_seq_number']
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])

    cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    ohe = OneHotEncoder(handle_unknown='ignore')
    ohe_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    train_ohe = ohe.fit_transform(df[ohe_cols])

    X_train = hstack([train_ohe, df[num_cols]])
    y_train = df['deal_probability']

    ohe_features = list(ohe.get_feature_names_out(ohe_cols))
    feature_names = ohe_features + num_cols

    return X_train, y_train, num_cols, num_imputer, scaler, cat_cols, cat_imputer, ohe_cols, ohe, feature_names



def transform_linear_pipeline(df_valid, num_cols, num_imputer, scaler, cat_cols, cat_imputer, ohe_cols, ohe):

    df = df_valid.copy()

    df[num_cols] = num_imputer.transform(df[num_cols])
    df[num_cols] = scaler.transform(df[num_cols])
    df[cat_cols] = cat_imputer.transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)

    df['image_top_1'] = df['image_top_1'].astype(str)

    valid_ohe = ohe.transform(df[ohe_cols])

    X_valid = hstack([valid_ohe, df[num_cols]])
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid


    

Разом протестируем модели и посмотрим на их средний RMSE. 

Тестировать всё буду на кросс-валидации с учетом того, что данные имеют временную метку (TimeSeriesSplit)

In [ ]:
def evaluate_model_cv(model, name, df, n_splits=5):
    print(f"-------------------{name}-------------------")
    print()
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    
    df = df.sort_values('activation_date').reset_index(drop=True)
    for train_idx, val_idx in tscv.split(df):
        df_tr, df_val = df.iloc[train_idx], df.iloc[val_idx]

        X_tr, y_tr, num_cols, num_imputer, scaler, cat_cols, cat_imputer, ohe_cols, ohe, feature_names = fit_linear_pipeline(df_tr)
        X_vl, y_vl = transform_linear_pipeline(df_val, num_cols, num_imputer, scaler, cat_cols, cat_imputer, ohe_cols, ohe)
            
        model.fit(X_tr, y_tr)

        preds = model.predict(X_vl)
        preds = np.clip(preds, 0, 1)

        fold_rmse = root_mean_squared_error(y_vl, preds)
        scores.append(fold_rmse)
        print(f"Получили RMSE на фолде: {fold_rmse}")
    rmse = np.mean(scores)
    print(f"-------------------Среднее RMSE: {np.mean(scores)}-------------------")    
    return rmse

In [ ]:
models={
    "Ridge":Ridge(alpha=0.1, solver='sparse_cg'),
    "SVM": LinearSVR(C=0.1, random_state=RANDOM_STATE),
    "Decision Tree":DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest":RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
}
results=[]
for name, model in models.items():
    rmse = evaluate_model_cv(model, name, df_small_train, n_splits=5)
    results.append({"model": name, "rmse": rmse})
pd.DataFrame(results).sort_values("rmse")

-------------------Ridge-------------------

Получили RMSE на фолде: 0.24491061065292052
Получили RMSE на фолде: 0.2420516088330892
Получили RMSE на фолде: 0.23635855713385837
Получили RMSE на фолде: 0.2350706311261202
Получили RMSE на фолде: 0.2380420328550912
-------------------Среднее RMSE: 0.2392866881202159-------------------
-------------------SVM-------------------



/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Получили RMSE на фолде: 0.2651885860584458


/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Получили RMSE на фолде: 0.2646766523100839


/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


... (2 outputs left out)

Получили RMSE на фолде: 0.258129927216759


/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Получили RMSE на фолде: 0.2636457203033029
-------------------Среднее RMSE: 0.2618833724902599-------------------
-------------------Decision Tree-------------------

Получили RMSE на фолде: 0.31871704846184495
Получили RMSE на фолде: 0.31227030943417433
Получили RMSE на фолде: 0.31093449192978534
Получили RMSE на фолде: 0.3126928466257903
Получили RMSE на фолде: 0.31702036396745786
-------------------Среднее RMSE: 0.31432701208381053-------------------
-------------------Random Forest-------------------

Получили RMSE на фолде: 0.24587883340220612
Получили RMSE на фолде: 0.24359623848902548
Получили RMSE на фолде: 0.23879659568285966
Получили RMSE на фолде: 0.23862764166822453
Получили RMSE на фолде: 0.24167321607862907
-------------------Среднее RMSE: 0.24171450506418896-------------------


,model,rmse
0,Ridge,0.239287
3,Random Forest,0.241715
1,SVM,0.261883
2,Decision Tree,0.314327


## Бустинг
- XGBoost
- LightGbm
- CatBoost

Тут уже не получится совместить препроцессинг для всех, так что для каждой модли сделаю свой

## XGBoost

In [ ]:
def fit_xgboost_pipeline(df_train):
    df = df_train.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'item_seq_number']


    # for col in cat_cols:
    #     df[col] = df[col].astype('category')
    
    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
    df[cat_cols] = encoder.fit_transform(df[cat_cols])
    X_train = df[feature_names]
    y_train = df['deal_probability']

    return X_train, y_train, feature_names, encoder

def transform_xgboost_pipeline(df_valid, feature_names, encoder):
    df = df_valid.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    # for col in cat_cols:
    #     df[col] = df[col].astype('category')
    df[cat_cols] = encoder.transform(df[cat_cols])

    X_valid = df[feature_names]
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
def evaluate_xgboost_model_cv(df, model, name, n_splits=5):
    print(f"-------------------{name}-------------------")
    print()
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    df = df.sort_values('activation_date').reset_index(drop=True)
    for train_idx, val_idx in tscv.split(df):
        df_tr, df_val = df.iloc[train_idx], df.iloc[val_idx]

        n = len(df_tr)
        cut = int(n * 0.9)
        df_tr_fit, df_tr_es = df_tr.iloc[:cut], df_tr.iloc[cut:]

        X_tr, y_tr, feature_names, encoder = fit_xgboost_pipeline(df_tr_fit)
        X_vl, y_vl = transform_xgboost_pipeline(df_val, feature_names, encoder)
        X_eval, y_eval = transform_xgboost_pipeline(df_tr_es, feature_names, encoder)

        model.fit(X_tr, y_tr, eval_set=[(X_eval, y_eval)])

        preds = model.predict(X_vl)
        preds = np.clip(preds, 0, 1)

        fold_rmse = root_mean_squared_error(y_vl, preds)
        scores.append(fold_rmse)
        print(f"Получили RMSE на фолде: {fold_rmse}")
    rmse = np.mean(scores)
    print(f"-------------------Среднее RMSE: {np.mean(scores)}-------------------")    
    return rmse, name

In [ ]:
rmse, name = evaluate_xgboost_model_cv(df=df_small_train, model=XGBRegressor(
    learning_rate=0.1, 
    missing=np.nan,    
    early_stopping_rounds=50, 
    random_state=RANDOM_STATE, 
    n_jobs=-1
), name='XGBoost')
results.append({"model": name, "rmse": rmse})

-------------------XGBoost-------------------

[0]	validation_0-rmse:0.25711
[1]	validation_0-rmse:0.25398
[2]	validation_0-rmse:0.25140
[3]	validation_0-rmse:0.24929
[4]	validation_0-rmse:0.24752
[5]	validation_0-rmse:0.24595
[6]	validation_0-rmse:0.24472
[7]	validation_0-rmse:0.24364
[8]	validation_0-rmse:0.24273
[9]	validation_0-rmse:0.24185
[10]	validation_0-rmse:0.24131
[11]	validation_0-rmse:0.24065
[12]	validation_0-rmse:0.24011
[13]	validation_0-rmse:0.23965
[14]	validation_0-rmse:0.23931
[15]	validation_0-rmse:0.23900
[16]	validation_0-rmse:0.23859
[17]	validation_0-rmse:0.23831
[18]	validation_0-rmse:0.23794
[19]	validation_0-rmse:0.23765
[20]	validation_0-rmse:0.23745
[21]	validation_0-rmse:0.23727
[22]	validation_0-rmse:0.23711
[23]	validation_0-rmse:0.23698
[24]	validation_0-rmse:0.23684
[25]	validation_0-rmse:0.23658
[26]	validation_0-rmse:0.23652
[27]	validation_0-rmse:0.23634
[28]	validation_0-rmse:0.23622
[29]	validation_0-rmse:0.23610
[30]	validation_0-rmse:0.23601
[3

## LightGBM

In [ ]:
def fit_lightgbm_pipeline(df_train):
    df = df_train.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'item_seq_number']
    
    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_cols:
        df[col] = df[col].astype('category')
    
    X_train = df[feature_names]
    y_train = df['deal_probability']

    return X_train, y_train, feature_names


def transform_lightgbm_pipeline(df_valid, feature_names):
    df = df_valid.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_cols:
        df[col] = df[col].astype('category')

    X_valid = df[feature_names]
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
def evaluate_lightgbm_model_cv(df, model, name, n_splits=5):
    print(f"-------------------{name}-------------------")
    print()
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    df = df.sort_values('activation_date').reset_index(drop=True)
    for train_idx, val_idx in tscv.split(df):
        df_tr, df_val = df.iloc[train_idx], df.iloc[val_idx]

        n = len(df_tr)
        cut = int(n * 0.9)
        df_tr_fit, df_tr_es = df_tr.iloc[:cut], df_tr.iloc[cut:]

        X_tr, y_tr, feature_names = fit_lightgbm_pipeline(df_tr_fit)
        X_vl, y_vl = transform_lightgbm_pipeline(df_val, feature_names)
        X_eval, y_eval = transform_lightgbm_pipeline(df_tr_es, feature_names)

        model.fit(X_tr, y_tr, eval_set=[(X_eval, y_eval)])

        preds = model.predict(X_vl)
        preds = np.clip(preds, 0, 1)

        fold_rmse = root_mean_squared_error(y_vl, preds)
        scores.append(fold_rmse)
        print(f"Получили RMSE на фолде: {fold_rmse}")
    rmse = np.mean(scores)
    print(f"-------------------Среднее RMSE: {np.mean(scores)}-------------------")    
    return rmse, name

In [ ]:
rmse, name = evaluate_lightgbm_model_cv(df=df_small_train, model=LGBMRegressor(
        learning_rate=0.1, 
        early_stopping_rounds=50, 
        random_state=RANDOM_STATE, 
        n_jobs=-1
    ), name='LightGBM')
results.append({"model": name, "rmse": rmse})

-------------------LightGBM-------------------

[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000256 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3183
[LightGBM] [Info] Number of data points in the train set: 36000, number of used features: 10
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Info] Start training from score 0.138178
Training until validation scores don't improve 

/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Early stopping, best iteration is:
[24]	valid_0's l2: 0.055313
Получили RMSE на фолде: 0.23521158796648098
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000396 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3895
[LightGBM] [Info] Number of data points in the train set: 72000, number of used features: 10
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Info] Start training from sco

/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Early stopping, best iteration is:
[44]	valid_0's l2: 0.0537175
Получили RMSE на фолде: 0.2341966449397352
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000542 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3969
[LightGBM] [Info] Number of data points in the train set: 108000, number of used features: 10
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Info] Start training from sc

/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Did not meet early stopping. Best iteration is:
[56]	valid_0's l2: 0.0539923
Получили RMSE на фолде: 0.23029573075560664
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000711 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4008
[LightGBM] [Info] Number of data points in the train set: 144000, number of used features: 10
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Info] Start tr

/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Did not meet early stopping. Best iteration is:
[68]	valid_0's l2: 0.0518759
Получили RMSE на фолде: 0.22896049557313863
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50


/opt/miniconda3/envs/studyenv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000887 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4020
[LightGBM] [Info] Number of data points in the train set: 180000, number of used features: 10
[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[LightGBM] [Info] Start training from score 0.138765
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[96]	valid_0's l2: 0.0529183
Получили RMSE на фолде: 0.23165841765244274
-------------------Среднее RMSE: 0.23206457537

## CatBoost

In [ ]:
def fit_catboost_pipeline(df_train):
    df = df_train.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_cols:
        df[col] = df[col].fillna('missing')

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'item_seq_number']

    X_train = df[feature_names]
    y_train = df['deal_probability']

    return X_train, y_train, feature_names


def transform_catboost_pipeline(df_valid, feature_names):
    df = df_valid.copy()

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    cat_cols = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1']
    for col in cat_cols:
        df[col] = df[col].fillna('missing')
        
    X_valid = df[feature_names]
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
def evaluate_catboost_model_cv(df, model, name, n_splits=5):
    print(f"-------------------{name}-------------------")
    print()
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    df = df.sort_values('activation_date').reset_index(drop=True)
    for train_idx, val_idx in tscv.split(df):
        df_tr, df_val = df.iloc[train_idx], df.iloc[val_idx]

        n = len(df_tr)
        cut = int(n * 0.9)
        df_tr_fit, df_tr_es = df_tr.iloc[:cut], df_tr.iloc[cut:]

        X_tr, y_tr, feature_names = fit_catboost_pipeline(df_tr_fit)
        X_vl, y_vl = transform_catboost_pipeline(df_val, feature_names)
        X_eval, y_eval = transform_catboost_pipeline(df_tr_es, feature_names)

        model.fit(X_tr, y_tr, eval_set=[(X_eval, y_eval)])

        preds = model.predict(X_vl)
        preds = np.clip(preds, 0, 1)

        fold_rmse = root_mean_squared_error(y_vl, preds)
        scores.append(fold_rmse)
        print(f"Получили RMSE на фолде: {fold_rmse}")
    rmse = np.mean(scores)
    print(f"-------------------Среднее RMSE: {np.mean(scores)}-------------------")    
    return rmse, name

In [ ]:
rmse, name = evaluate_catboost_model_cv(df=df_small_train, model=CatBoostRegressor(
        learning_rate=0.1, 
        verbose=False, 
        early_stopping_rounds=50, 
        cat_features=['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'],
        random_seed=RANDOM_STATE
    ), name='CatBoost')
results.append({"model": name, "rmse": rmse})

-------------------CatBoost-------------------

Получили RMSE на фолде: 0.23287594620495175
Получили RMSE на фолде: 0.2349281929884233
Получили RMSE на фолде: 0.23110945546748307
Получили RMSE на фолде: 0.22785660356585033
Получили RMSE на фолде: 0.23108610185765868
-------------------Среднее RMSE: 0.23157126001687342-------------------


## Сравнительная таблица с показателями моделей

| № | Модель | RMSE |
| :--- | :--- | :---: |
| 1 | **CatBoost** | **0.2315713** |
| 2 | LightGBM | 0.2320646 |
| 3 | XGBoost | 0.2335173 |
| 4 | Ridge | 0.2392870 |
| 5 | Random Forest | 0.2417150 |
| 6 | SVM | 0.2618830 |
| 7 | Decision Tree | 0.3143270 |


CatBoost победил, что неудивительно!

Дальше будем работать именно с ним, постараемся уменьшить RMSE=0.2316 засчет новых признаков и подбора ГП